In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import scipy as sp

# Data
Import and engineer the data into arrays to be used as inputs and labels for the neural network.

In [2]:
# import data
train = pd.read_csv("C:/Users/Grego/Downloads/train.csv") # training data with inputs and labels

In [3]:
# turn into ndarray
train = np.array(train)

# one-hot encode the labels
all_labels = np.zeros((max(train[:, 0]) + 1, train.shape[0])) # make an array of zeros
all_labels[train[:, 0], np.arange(all_labels.shape[1])] = 1 # one-hot encode the labels
all_inputs = train[:, 1:].T/256


# get data for the data with labels 0 or 1
zero_one = train[train[:, 0] <= 1]
zero_one_labels = (np.ones(zero_one.shape[0]) - zero_one[:, 0]).reshape(1, -1) # labels
zero_one_inputs = zero_one[:, 1:].T/256 # inputs - each column corresponds to a data point

In [18]:
# select input and output
k = 10000
in_ = zero_one_inputs[:, :]
out_ = zero_one_labels[:, :]

# Cost and Loss Functions

In [5]:
def mean_squared_error_cost(y, y_hat):
    # y : ndarray for one training label with shape (n, m) 
    # y_hat : ndarray for one training example with shape (n, m)
    # n : number of neurons in output layer
    # m : number of training examples
    
    n, m = y.shape
    loss = []
    for i in range(m):
        loss.append(np.dot(y[:, i] - y_hat[:, i], y[:, i] - y_hat[:, i]))
    
    # return loss
    return np.mean(loss)
mean_squared_error_cost(out_, out_)

0.0

In [6]:
# this function will return the loss if a single training example is passed and will return the cost if multiple examples are passed
def cross_entropy_cost(y, y_hat):
    # y : ndarray for one training label with shape (n, m) 
    # y_hat : ndarray for one training example with shape (n, m)
    # n : number of neurons in output layer
    # m : number of training examples
    
    n,m = y.shape
    
    # create loss array
    loss = np.zeros_like(y, dtype = np.float64)
    
    # calculate loss values for each entry
    for j in range(m):
        for i in range(n):
            loss[i, j] = y[i, j]*np.log(max(y_hat[i, j], 10**(-7)), dtype = np.float64) + (1 - y[i, j])*np.log(max(1-y_hat[i, j], 10**(-7)), dtype = np.float64)
    
    # take the average of the loss array and return loss
    return - np.mean(loss)
#cross_entropy_cost(out_, np.random.rand(1, out_.size))

# Activation Functions

In [7]:
def linear(z):
    # z : ndarray with shape (n,1):
    
    # return z
    return z

In [8]:
def relu(z):
    # z : ndarray with shape (n,1):
    
    c = np.zeros_like(z, dtype = np.float64)
    r = np.maximum(c, z)
    
    # return r
    return r

In [9]:
def sigmoid(z):
    # z : ndarray with shape (n,1):

    s = 1/(1 + np.exp(-z))
    
    # return r
    return s

In [10]:
def softmax(z):
    # z : ndarray with shape (n,1):
    
    s = np.exp(z)/max(np.sum(np.exp(z)), 10**(-7))
    
    # return r
    return s

# Network Building

In [11]:
# initialize weights and biases for a neural network
def init_weights_biases(layer_lengths):
    # layer_lengths : ndarray with shape (n,) with each element being the number of neurons in that layer
    
    # create dicts to store ndarrays for weights and biases
    weights, biases = {}, {}
    
    # create ndarrays for the weights and biases of each layer
    for i in range(1, layer_lengths.size):
        weights['layer' + str(i)] = np.random.randn(layer_lengths[i], layer_lengths[i - 1])#np.zeros((layer_lengths[i], layer_lengths[i - 1]), dtype = np.float64)
        biases['layer' + str(i)] = np.zeros((layer_lengths[i], 1))#np.full((layer_lengths[i], 1), 0.5, dtype = np.float64)#np.random.rand(layer_lengths[i], 1)
    
    # return dicts for weights and biases
    return weights, biases

In [12]:
# calculate the neuron activations for the network for a single/multiple training examples
def activations_multiple(x, weights, biases, f):
    # x : ndarray with shape (n, m) for with n being the number of input neurons, m being the number of datapoints
    # weights : dict containing ndarrays of weights each with shape (n,m) for some n,m
    # biases : dict containing ndarrays of biases each with shape (n,1) for some n
    # f : ndarray with shape (m,) containing the activation functions for each layer
    
    # create dicts to store activation and z_activation values
    activations, z_activations = {}, {}
    
    n = len(weights) + 1
    
    # set layer
    layer = x
    
    # set activations['layer0'] to the input layer
    activations['layer0'] = x
    
    # calculate activations
    for i in range(1, n):
        z_activations['layer' + str(i)] = np.matmul(weights['layer' + str(i)], layer) + biases['layer' + str(i)]
        activations['layer' + str(i)] = f[i - 1](z_activations['layer' + str(i)])
        layer = activations['layer' + str(i)]
    
    # return dicts for activations and z_activations
    return activations, z_activations
    

# Gradient Descent

In [13]:
# computes the gradients for a single data point
def backpropagation_single(x, y, weights, biases, cost, alpha, delta, f, activations, z_activations):
    # x : ndarray with shape (n, 1) for inputs for some n
    # y : ndarray with shape (n, 1) for labels for some n
    # weights : dict containing ndarrays of weights each with shape (n,m) for some n,m
    # biases : dict containing ndarrays of biases each with shape (n,1) for some n
    # cost : function taking ndarrays as inputs
    
    # initialize the derivative dictionaries to store the ndarrays
    q = len(weights)
    djda = {} # column vectors
    djdz = {} # column vectors
    djdw = {} # matrices
    djdb = {} # column vectors
    
    # initialize the intermediate derivative dictionaries (these will be used for calculations)
    dadz = {} # column vectors
    dzdw = {} # matrices
    dzdb = {} # column vectors
    dzda = {} # matrices
    
    # calculate activations and z_activations for the network - only needed if activations and z_activations are not passed as inputs
    #activations, z_activations = activations_multiple(x, weights, biases, f)
    
    # calculate djda for each neuron in the output layer
    djda['layer' + str(q)] = np.empty_like(activations['layer' + str(q)], dtype = np.float64)
    
    for i in range(y.shape[0]):
        # turn individual neurons in the output layer into arrays to be fed into cost function
        y_array = np.array(y[i, 0]).reshape(1,1)
        y_hat_array = np.array(activations['layer' + str(q)][i, 0], dtype = np.float64).reshape(1,1)
        djda['layer' + str(q)][i, 0] = (cost(y_array, y_hat_array + delta) - cost(y_array, y_hat_array))/delta
        #print((cost(y_array, y_hat_array + delta) - cost(y_array, y_hat_array))/delta, "check")
    
    # find intermediate derivatives going backwards
    for i in range(q, 0, -1):
        # find dadz
        dadz['layer' + str(i)] = (f[i - 1](z_activations['layer' + str(i)] + delta) - f[i - 1](z_activations['layer' + str(i)]))/delta
        
        # find dzdw, dzdb, dzda
        dzdw['layer' + str(i)] = np.repeat(activations['layer' + str(i - 1)].T, weights['layer' + str(i)].shape[0], axis = 0)
        dzdb['layer' + str(i)] = np.ones_like(biases['layer' + str(i)], dtype = np.float64)
        dzda['layer' + str(i - 1)] = weights['layer' + str(i)].T # each row k of this 2-D array houses the derivative of neuron k with respect to that z_neuron
        
        # calculate djdw, djdb, djda
        djdz['layer' + str(i)] = djda['layer' + str(i)]*dadz['layer' + str(i)]
        djdw['layer' + str(i)] = djdz['layer' + str(i)]*dzdw['layer' + str(i)]
        djdb['layer' + str(i)] = djdz['layer' + str(i)]*dzdb['layer' + str(i)]
        djda['layer' + str(i - 1)] = np.matmul(dzda['layer' + str(i - 1)], djdz['layer' + str(i)])
        
    return djdw, djdb, djda, djdz, dadz, dzdw, dzdb, dzda

In [19]:
x_ = in_#[:, 0].reshape(-1, 1)
y_ = out_#[:, 0].reshape(-1, 1)
w_, b_ = init_weights_biases(np.array([x_.shape[0], 200, 10]))
cost_ = mean_squared_error_cost
alpha_ = 0.01
delta_ = 0.01
f_ = np.array([relu, sigmoid])
epochs_ = 10

In [20]:
# This function will use the gradient() function defined above to calculate the gradients of the cost w.r.t. the parameters for
# a given training example and then take a step in that direction using the learning rate alpha.
def sgd_steps(x, y, weights, biases, f, cost, delta, alpha):
    # for each observation, calculate the gradient and update the weights and biases accordingly.
    
    # create dictionaries to store history
    cost_history = {}
    weights_history = {}
    biases_history = {}
    
    # create dictionaries for each update 
    for i in range(x.shape[1]):
        weights_history['update' + str(i)] = {}
        biases_history['update' + str(i)] = {}
    
    # split data into dictionaries containing the activations and z_activations
    layers, z_layers = {}, {}
    
    # calculate activations for all the inputs
    aaz = activations_multiple(x, weights, biases, f)
    
    # create dictionaries to store activations and z_activations for each observation
    for i in range(x.shape[1]):
         # create dictionaries to store activations and z_activations for each observation
        layers['observation' + str(i)], z_layers['observation' + str(i)] = {}, {}
        
        # grab the activation layer columns corresponding to observation i
        for j in range(len(aaz[0])):
            layers['observation' + str(i)]['layer' + str(j)] = aaz[0]['layer' + str(j)][:, i].reshape(-1, 1)
        
        # grab the z_activation layer columns corresponding to observation i
        for j in range(1, len(aaz[1]) + 1):
            z_layers['observation' + str(i)]['layer' + str(j)] = aaz[1]['layer' + str(j)][:, i].reshape(-1, 1)
    
    # update weights and biases for each training example
    for i in range(len(layers)):
        q = len(weights)
        
        # calculate gradients
        djdw, djdb, djda, djdz, dadz, dzdw, dzdb, dzda = backpropagation_single(x[:, i].reshape(-1, 1), y[:, i].reshape(-1, 1), weights, biases, cost, alpha, delta, f, layers['observation' + str(i)], z_layers['observation' + str(i)])
        
        # update weights and biases by layer
        for j in range(1, q + 1):
            # update and save weights
            weights['layer' + str(j)] = weights['layer' + str(j)] - alpha*djdw['layer' + str(j)]
            weights_history['update' + str(i)]['layer' + str(j)] = weights['layer' + str(j)]
            
            # update and save biases
            biases['layer' + str(j)] = biases['layer' + str(j)] - alpha*djdb['layer' + str(j)]
            biases_history['update' + str(i)]['layer' + str(j)] = biases['layer' + str(j)]
        
        # print cost for every 100th update
        if i % 1000 == 0:
            # calculate and record cost
            temp_aaz = activations_multiple(x, weights, biases, f)
            temp_cost = cost(y, temp_aaz[0]['layer' + str(q)])
            cost_history['update' + str(i)] = temp_cost
            
            print(temp_cost, 'update' + str(i))
            #print((temp_aaz[0]['layer' + str(q)] - y[:, i].reshape(-1, 1))[:, 0])
            #print(djda)
            #if temp_cost < 1:
            #    break
            
        
    return weights, biases, weights_history, biases_history, cost_history, layers, z_layers

In [21]:
w_, b_, w_history_, b_history_, cost_history_, layers_, z_layers_ = sgd_steps(x_, y_, w_, b_, f_, cost_, delta_, alpha_)

5.599904494845195 update0
5.492247390244413 update1000
5.484947529211692 update2000
5.1674812267485875 update3000
5.091788287810536 update4000
5.09414669588909 update5000
5.317334098689256 update6000
5.295688235612305 update7000
5.25973156059069 update8000


In [17]:
bp = []
for i in b_history_:
        bp.append(b_history_[i]['layer1'][1,0])
plt.plot(bp)


KeyboardInterrupt



In [ ]:
wp = []
for i in w_history_:
        wp.append(w_history_[i]['layer2'][0,4])
plt.plot(wp)

In [ ]:
def gradient_descent(x, y, weights, biases, f, cost, delta, alpha, epochs):
    for i in range(epochs):
        sgd_steps(x, y, weights, biases, f, cost, delta, alpha)
        y_hat = activations_multiple(x_, w_, b_, f_)[0]['layer' + str(len(weights))]
        print(f'Epoch {i} training accuracy : {success_rate(y_hat, y) :.3f}')

In [19]:
gradient_descent(x_, y_, w_, b_, f_, cost_, delta_, alpha_, epochs_)

1.0000958669793558 update0
1.0000958669793558 update1000
1.0000958669793556 update2000
1.0000958669793554 update3000



KeyboardInterrupt



In [37]:
y_hat_ = activations_multiple(x_, w_, b_, f_)[0]['layer' + str(len(w_))]
y_hat_

C:\Users\Grego\AppData\Local\Temp\ipykernel_948\2260294278.py:4: RuntimeWarning: overflow encountered in exp
  s = 1/(1 + np.exp(-z))


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [26]:
# This function will return the proportion of examples correctly predicted
def success_rate(y, y_hat):
    # y : ndarray - One-Hot encoded true values
    # y_hat : ndarray - neural network output of the same shape as y
    
    # convert y and y_hat into number label from 0-9
    a = np.argmax(y, axis = 1)
    a_hat = np.argmax(y_hat, axis = 1)
    
    # calculate and return the proportion of correctly labelled examples
    p = a[a == a_hat].size/a.size
    return p

In [38]:
success_rate(y_, y_hat_)

0.1

In [112]:
a_ = np.argmax(y_, axis = 1)
a_hat_ = np.argmax(y_hat_, axis = 1)

In [115]:
p = a_[a_ == a_hat_].size/a_.size
p

0.0

In [78]:
np.argmax(np.array([[1,2,3], [4,5,6]]), axis = 0)

array([1, 1, 1], dtype=int64)